# 31 — Parsing Error Handling
**Goal:** Gracefully handle corrupt files, empty documents, and edge cases.

Chapters 26–30 assumed the input is a valid PDF, DOCX, or image. Real resume uploads are not that polite: truncated downloads, password-protected files, 0-byte uploads, and mislabeled extensions happen constantly. This chapter hardens the extraction layer so a single bad file degrades to a warning instead of crashing a batch run.

**Why it matters for resumes / ATS:** a parsing pipeline that throws on one corrupt resume stops the whole batch — every candidate after it goes unprocessed. Error handling is not boilerplate here; it is what separates a demo script from a service you can point at a production inbox.

## 1. Common Failure Modes

The failures cluster at different layers, which decides where you catch them. Container corruption (broken ZIP/PDF structure) fails at open; 0-byte and truncated files fail at read; password protection fails at decrypt; image-only PDFs fail at `extract_text()` with an *empty result, not an exception*; wrong extensions fail at dispatch — you hand a PDF parser a `.docx`; encoding errors fail at decode.

**What the code does:** prints the seven failure classes as a checklist — corrupt PDF/DOCX, empty files, password-protected documents, image-only files, wrong extension, truncated downloads, non-UTF8 content.

**Expected output:** the bullet list of all seven. Notice the shape: some modes raise, some return empty text, some return garbage — so a robust parser must handle all three outcomes, not just exceptions. That asymmetry drives the design in section 3.

In [ ]:
faults = [
    "Corrupt PDF/DOCX (broken ZIP structure)",
    "Empty files (0 bytes)",
    "Password-protected documents",
    "File with only images (no text layer)",
    "Wrong extension (.pdf but is .docx)",
    "Truncated downloads",
    "Encoding errors (non-UTF8 content)",
]
for f in faults: print(f"  - {f}")

## 2. Magic Byte File Detection

Extensions are untrusted input — anyone can name a `.docx` file `.pdf`. Magic bytes are the file's own signature: `%PDF` for PDF, `PK\x03\x04` for any ZIP-based format (`.docx`, `.xlsx`, `.pptx`), `\x89PNG` for PNG, `\xff\xd8\xff` for JPEG. `detect_type()` reads the first 4 bytes and matches prefixes against that table.

**What the code does:**
- Writes a fake `%PDF-1.4` payload to a temp file whose *name* ends in `.pdf`
- Runs `detect_type()` and prints the detected type next to the extension

**Expected (verified):** `Detected: pdf (vs extension: .pdf)` — trivially matching here, but the same check catches the real danger: a `.pdf`-named file whose bytes start `PK\x03\x04` (a ZIP, i.e. actually a DOCX) reports `docx`. That mismatch tells you the parser must be chosen by *content*, not name. Four bytes are enough for these signatures; other formats need longer prefixes.

In [ ]:
def detect_type(filepath):
    sigs = {b'%PDF': 'pdf', b'PK\x03\x04': 'docx', b'\x89PNG': 'png', b'\xff\xd8\xff': 'jpg'}
    with open(filepath, 'rb') as f:
        h = f.read(4)
    for sig, t in sigs.items():
        if h.startswith(sig): return t
    return 'unknown'

import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as f:
    f.write(b'%PDF-1.4 fake')
    fn = f.name
print(f"File: {fn}")
print(f"Detected: {detect_type(fn)} (vs extension: .pdf)")
os.unlink(fn)

## 3. Robust Document Parser with Fallbacks

The parser as a **strategy chain**: register extractors in order of preference, try them one by one, keep the first one that returns non-empty text, and log-and-continue on failure. If every strategy fails, return `""` — a defined fallback rather than a crash. This mirrors the Ch. 26–28 tool stack: `pdfplumber` → PyMuPDF → OCR is the natural chain, each step slower but more tolerant than the last.

**What the code does:**
- `SafeParser` holds an ordered `strategies` list; `add(name, fn)` appends an extractor
- `parse(filepath)` loops the strategies, catches any `Exception` per strategy (printing `name failed: ...`), and returns the first non-empty result
- Registers three stub extractors and calls `parse("test.pdf")`

**Expected:** `Result: 'extracted pdf text'` — the first strategy succeeds immediately. With the stubs replaced by real parsers, a corrupt PDF prints `pdfplumber failed: ...`, then `PyMuPDF failed: ...`, then falls through to OCR or `""`. Note the guard `if text and text.strip()` — it treats *empty* output as failure too, catching the image-only-PDF case that raises nothing.

In [ ]:
class SafeParser:
    def __init__(self):
        self.strategies = []
    def add(self, name, fn):
        self.strategies.append((name, fn))
    def parse(self, filepath):
        for name, fn in self.strategies:
            try:
                text = fn(filepath)
                if text and text.strip():
                    return text
            except Exception as e:
                print(f"  {name} failed: {e}")
        return ""

p = SafeParser()
p.add("pdfplumber", lambda f: "extracted pdf text")
p.add("PyMuPDF", lambda f: "text from pymupdf")
p.add("OCR", lambda f: "text from ocr")
print(f"Result: '{p.parse('test.pdf')}'")

## Summary: Always validate file types by magic bytes, not extension. Layer fallbacks.

**Assume every file is hostile until proven otherwise.** Validate by magic bytes, not extension; treat empty output as failure just like exceptions; and chain fallback strategies (PDF → PyMuPDF → OCR) so a single corrupt file costs one warning, not the whole batch. Defined fallbacks — `""` for text, `"unknown"` for type — keep downstream stages from crashing on missing data.

This closes the extraction half of the pipeline: from file bytes (Ch. 26–28) through cleaning (Ch. 29) and routing (Ch. 30), everything now produces *clean, language-aware text* — or a graceful failure. The next chapter finally gives that text meaning: Ch. 32, Section Detection, turns it into a structured profile.